# Radionica 6: RAG i izgradnja baze znanja

**GirlTHing AI Lab — Blok 2**

> Ćelije označene sa ✏️ **ZADATAK** popunjavate sami. Ostale ćelije samo pokrenite i pročitajte rezultat.

## Ciljevi radionice

1. Razumjeti zašto LLM sam nije dovoljan i šta rješava **RAG** (Retrieval-Augmented Generation).
2. Pretvoriti tekst u **embeddings** (vektore) i mjeriti sličnost.
3. Podijeliti dokumente na **chunkove**.
4. Napraviti **semantičku pretragu** nad bazom znanja.
5. Spojiti pretragu i model u RAG chatbot koji navodi izvore i zna reći "ne znam".

```
Pitanje → embedding → pronađi najsličnije chunkove → [chunkovi + pitanje] → LLM → odgovor s izvorima
```

In [ ]:
# Instalacija paketa (u Colabu je potrebno samo jednom po sesiji)
%pip install -q openai numpy pydantic python-dotenv pypdf

import os
from openai import OpenAI

# API ključ: u Colabu iz "Secrets" (ikona ključa lijevo), lokalno iz .env datoteke
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

client = OpenAI()
MODEL = "gpt-6-luna"     # jeftin i brz model, dovoljan za sve vježbe
print("Spremno ✅")

In [ ]:
# Preuzimanje materijala (dokumenti za bazu znanja i rag.py)
REPO_URL = "https://github.com/asja-b/girlthing-ai-lab-blok2"
FOLDER = "data_moje"     # <-- "data" za dokumente s radionice, "data_moje" za moje

import os, sys
from pathlib import Path

for candidate in [Path("."), Path(".."), Path("girlthing-ai-lab-blok2")]:
    if (candidate / FOLDER).is_dir() and (candidate / "rag.py").exists():
        ROOT = candidate.resolve()
        break
else:
    os.system(f"git clone -q {REPO_URL}")
    ROOT = Path("girlthing-ai-lab-blok2").resolve()

DATA_DIR = ROOT / FOLDER
sys.path.insert(0, str(ROOT))
print("Dokumenti:", sorted(p.name for p in DATA_DIR.iterdir()))

## 1. Problem: model ne zna naše podatke

In [ ]:
r = client.responses.create(model=MODEL, input="Kako se zove završni događaj programa GirlTHing AI Lab i kada se dijele ruksaci?")
print(r.output_text)

Model nije vidio naše dokumente, pa ili ne zna, ili izmišlja. Rješenje: prije pitanja **pronađemo relevantan tekst** i stavimo ga u prompt.

## 2. Embeddings: tekst kao vektor

Embedding model pretvara tekst u vektor od 1.536 brojeva. Tekstovi **sličnog značenja** imaju vektore koji pokazuju u sličnom smjeru, čak i kad ne dijele iste riječi.

Sličnost mjerimo **kosinusnom sličnošću** (sjećate se vektora iz Bloka 1):

$$\text{cos}(a, b) = \frac{a \cdot b}{\|a\| \, \|b\|}$$

Vrijednost blizu 1 znači vrlo slično, a blizu 0 nepovezano.

In [ ]:
import numpy as np

EMBED_MODEL = "text-embedding-3-small"

def embed(texts, batch_size=100):
    # Moja baza ima preko 3000 chunkova, a jedan poziv ne prima toliko ulaza odjednom,
    # pa saljem u serijama. rag.py radi isto, tamo stoji batch_size=100.
    vectors = []
    for i in range(0, len(texts), batch_size):
        resp = client.embeddings.create(model=EMBED_MODEL, input=texts[i:i + batch_size])
        vectors.extend(d.embedding for d in resp.data)
        if len(texts) > batch_size:
            print(f"  {min(i + batch_size, len(texts))}/{len(texts)}", end="\r")
    return np.array(vectors)

recenice = [
    "Kako da se pripremim za intervju?",
    "Savjeti za razgovor za posao",
    "Koliko dug treba biti CV?",
    "Recept za čokoladnu tortu",
]
V = embed(recenice)
print("Oblik matrice:", V.shape)

### ✏️ ZADATAK 1
Implementiraj funkciju `cosine_similarity(a, b)` pomoću numpy-ja i ispiši sličnost prve rečenice sa svim ostalima.

In [ ]:
def cosine_similarity(a, b):
    # skalarni proizvod podijeljen umnoskom duzina oba vektora
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

for i in range(1, len(recenice)):
    print(f"{cosine_similarity(V[0], V[i]):.3f}  {recenice[0]}  \u2194  {recenice[i]}")

💡 "intervju" i "razgovor za posao" nemaju zajedničkih riječi, a ipak su najsličniji. Ovo je **semantička** pretraga, a ne pretraga po ključnim riječima.

## 3. Učitavanje dokumenata

### Provjera prije učitavanja

Dokumenti s radionice su bili `.md`, dakle čisti tekst. Moji su PDF-ovi, pa moram provjeriti
izvlači li se iz svake stranice tekst. Skenirana stranica je slika, `extract_text()` na njoj
vraća prazan string bez greške, pa dio baze tiho nestane.

In [ ]:
from pypdf import PdfReader

for path in sorted(DATA_DIR.glob("*.pdf")):
    po_stranici = [p.extract_text() or "" for p in PdfReader(path).pages]
    prazne = [i for i, t in enumerate(po_stranici, 1) if len(t.strip()) < 30]
    print(f"{path.name:38} {len(po_stranici):3} str, bez teksta: {prazne if prazne else 'nema'}")

Dvanaest stranica nema teksta: 4 od 7 u `maksimalno-ucesce.pdf` i 8 od 20 u
`osnovni-paket.pdf`. To su skenirani prilozi s tabelama iznosa, uslikani pa zalijepljeni u PDF.

Probala sam OCR, ali nazivi usluga izlaze polovično a iznosi se gube ili izmisle, pa te
stranice ostavljam neizvučene. Za projekat čiji je smisao tačan citat, pogrešan broj uz citat
je gori od poštenog "nemam to u bazi".

**Posljedica:** bot može reći ko je oslobođen participacije, ali ne i koliko se plaća.

In [ ]:
from pypdf import PdfReader  # za PDF dokumente

def load_documents(folder):
    docs = []
    for path in sorted(Path(folder).glob("*")):
        if path.suffix in (".md", ".txt"):
            docs.append((path.name, path.read_text(encoding="utf-8")))
        elif path.suffix == ".pdf":
            text = "\n".join(p.extract_text() or "" for p in PdfReader(path).pages)
            docs.append((path.name, text))
    return docs

docs = load_documents(DATA_DIR)
for name, text in docs:
    print(f"{name:32} {len(text):5} znakova")

## 4. Chunking: dijeljenje na manje dijelove

Zašto ne šaljemo cijele dokumente?
- Pretraga je preciznija kad je svaki dio o **jednoj temi**.
- Manje teksta u promptu znači **manju cijenu** i manje "šuma" za model.

Najjednostavniji pristup: komadi fiksne dužine s **preklapanjem** (overlap), da rečenica na granici ne izgubi kontekst.

```
|-------- chunk 1 --------|
                  |-------- chunk 2 --------|
                                    |-------- chunk 3 --------|
                  <overlap>
```

### ✏️ ZADATAK 2
Napiši `chunk_text(text, size, overlap)` koja vraća listu komada dužine najviše `size` znakova, gdje svaki sljedeći počinje `size - overlap` znakova nakon prethodnog.

In [ ]:
def chunk_text(text, size=600, overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start:start + size])
        start += size - overlap   # pomak je manji od size, pa se komadi preklapaju
    return chunks

test = chunk_text("abcdefghij" * 20, size=50, overlap=10)
print(len(test), [len(c) for c in test])

In [ ]:
chunks, sources = [], []
for name, text in docs:
    for c in chunk_text(text):
        chunks.append(c)
        sources.append(name)

print("Ukupno chunkova:", len(chunks))
print("\nPrimjer chunka:\n", chunks[3])

🤔 **Diskusija:** chunk iznad možda počinje usred riječi. U `rag.py` koristimo pametniju verziju koja reže po **pasusima**. Kakvi su nedostaci premalih, a kakvi prevelikih chunkova?

## 5. Indeks i pretraga

Izračunamo embedding za svaki chunk jednom i spremimo ih u matricu. To je naša mala **vektorska baza**. Ako vektore normalizujemo na dužinu 1, kosinusna sličnost postaje običan skalarni proizvod, pa sve chunkove poredimo jednim množenjem matrice.

In [ ]:
chunk_vectors = embed(chunks)
chunk_vectors = chunk_vectors / np.linalg.norm(chunk_vectors, axis=1, keepdims=True)
print("Indeks:", chunk_vectors.shape)

### ✏️ ZADATAK 3
Napiši `search(query, k)` koja vraća `k` najsličnijih chunkova kao listu `(sličnost, izvor, tekst)`, sortiranu od najsličnijeg.

*Pomoć:* `np.argsort(-scores)[:k]` daje indekse `k` najvećih vrijednosti.

In [ ]:
def search(query, k=3):
    q = embed([query])[0]
    q = q / np.linalg.norm(q)
    scores = chunk_vectors @ q          # vektori su vec normalizovani -> ovo je kosinusna slicnost
    best = np.argsort(-scores)[:k]      # indeksi k najvecih, od najveceg
    return [(float(scores[i]), sources[i], chunks[i]) for i in best]

for score, src, text in search("Od koje godine se radi skrining grlica materice?"):
    print(f"{score:.3f}  [{src}]  {text[:80]}...")

In [ ]:
for q in ["Koliko dana traje normalan menstrualni ciklus?",
          "Koliko minuta aktivnosti sedmično preporučuje WHO?",
          "Imam li pravo na izbor doktora?",
          "Koja je najbolja pizzerija u Tuzli?"]:
    print("❓", q)
    for score, src, _ in search(q):
        print(f"   {score:.3f}  {src}")

💡 Pogledaj sličnosti za posljednje pitanje (pizzerija). Sve su niske, jer u bazi nema ničeg relevantnog. To koristimo da chatbot zna reći **"ne znam"**.

## 6. RAG: pretraga + generisanje

Pronađene chunkove ubacimo u prompt kao **KONTEKST** i tražimo od modela da odgovara samo na osnovu njega.

In [ ]:
RAG_INSTRUKCIJE = """Ti si asistent koji odgovara na pitanja o ženskom zdravlju i o pravima
u zdravstvenom sistemu Federacije BiH.
Odgovaraj ISKLJUČIVO na osnovu priloženog KONTEKSTA.
Ako odgovor nije u kontekstu, reci: "Nemam tu informaciju u svojoj bazi znanja."
Na kraju navedi izvore u obliku [naziv_datoteke].
Odgovaraj na bosanskom, kratko i jasno.

GRANICE:
- ne postavljaš dijagnozu i ne tumačiš simptome konkretne osobe
- ne daješ doze. Brojke iz dokumenata su populacione preporuke, ne doze za pojedinca,
  i to izričito kažeš kada ih navodiš
- ne daješ kalorijske ciljeve ni ciljnu težinu
- na opisan simptom kažeš šta se po smjernici provjerava i kod kojeg ljekara

Na kraju svakog odgovora stoji: ovo nije medicinski savjet i nije zamjena za ljekara."""

### ✏️ ZADATAK 4
Napiši `rag_answer(question)`:
1. pronađi 4 najsličnija chunka,
2. odbaci one sa sličnošću ispod `MIN_SCORE`,
3. složi kontekst u obliku `[izvor]\ntekst`, odvojen linijama `---`,
4. pozovi model s `RAG_INSTRUKCIJE` i vrati odgovor.

In [ ]:
MIN_SCORE = 0.25

def rag_answer(question):
    hits = search(question, k=4)
    hits = [h for h in hits if h[0] >= MIN_SCORE]        # 2. odbaci preslabe pogotke

    if not hits:                                          # nista relevantno -> ne zovemo model
        return "Nemam tu informaciju u svojoj bazi znanja."

    kontekst = "\n---\n".join(f"[{src}]\n{text}" for _, src, text in hits)   # 3. slozi kontekst

    r = client.responses.create(                           # 4. pozovi model
        model=MODEL,
        instructions=RAG_INSTRUKCIJE,
        input=f"KONTEKST:\n{kontekst}\n\nPITANJE: {question}",
    )
    return r.output_text

print(rag_answer("Od koje godine se preporučuje skrining grlića materice i u kojem intervalu?"))

In [ ]:
for q in ["Koliko dugo smije trajati menstrualno krvarenje?",
          "Moram li platiti papa test?",
          "Koliko miligrama željeza da uzmem?",          # brojka POSTOJI u bazi, ne smije je dati kao dozu
          "Koliko kalorija da jedem da smršam 5 kg?",    # mora odbiti
          "Koliko košta pregled kod specijaliste?"]:     # iznosi su na skeniranim stranicama
    print("❓", q)
    print("💬", rag_answer(q), "\n")

### ✏️ ZADATAK 5: Podešavanje praga
Isprobaj `MIN_SCORE` vrijednosti 0.1, 0.25 i 0.45 na pitanjima iznad. Šta se dešava kad je prag prenizak, a šta kad je previsok? Zapiši zaključak u ćeliju ispod.

*Moj zaključak:*

**MIN_SCORE = 0.1** je prenizak. Kroz filter prolaze i chunkovi koji s pitanjem nemaju
veze, pa model dobije šum i počne od njega graditi odgovor. Na pitanju o pizzeriji više
ne kaže da ne zna, nego sastavi nešto iz nepovezanih dokumenata. To je gora greška od
„ne znam", jer korisnica ne može primijetiti da je odgovor izmišljen.

**MIN_SCORE = 0.45** je previsok. Sada ispadaju i tačni chunkovi, pa bot kaže da nema
informaciju iako je odgovor u bazi. Baza je upotrebljiva, a izgleda prazno.

**0.25 je dobra sredina** za ovu bazu. Propusti tačne pogotke, a odbije očito nepovezane.

Ono što sam primijetila: prag nije univerzalan broj. Zavisi od embedding modela i od
veličine chunkova, pa se bira **mjerenjem na test setu**, a ne pogađanjem. Zato i pravimo
test set u domaćoj zadaći, uključujući pitanja na koja odgovora nema.

Primijetila sam i da moja baza traži oprezniji prag nego baza s radionice. Moji dokumenti su
dugi i pisani sličnim službenim jezikom, pa dosta chunkova djeluje međusobno srodno i dobija
osrednje sličnosti. Kod kratkih `.md` datoteka razlika između tačnog i netačnog pogotka bila
je oštrija.

## 7. Bonus: vektorska baza ChromaDB

Za veće baze (hiljade dokumenata) koristi se prava vektorska baza. ChromaDB je besplatna i radi lokalno. Logika je ista kao naša: spremi vektore, a zatim traži najsličnije.

In [ ]:
%pip install -q chromadb
import chromadb

chroma = chromadb.Client()                       # baza u memoriji
col = chroma.get_or_create_collection("studentski_asistent", metadata={"hnsw:space": "cosine"})
col.add(ids=[str(i) for i in range(len(chunks))],
        documents=chunks,
        embeddings=chunk_vectors.tolist(),
        metadatas=[{"source": s} for s in sources])

q = embed(["Kada se otvaraju prijave za stipendije?"])[0].tolist()
res = col.query(query_embeddings=[q], n_results=3)
for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
    print(f"{1 - dist:.3f}  [{meta['source']}]  {doc[:70]}...")

## 8. Sve zajedno: `rag.py`

Sve što smo danas napravili nalazi se u datoteci `rag.py`, uz nekoliko poboljšanja: chunking po pasusima, čuvanje indeksa u datoteci (embeddings se računaju samo jednom) i historiju razgovora. Tu datoteku koristimo na sljedećoj radionici za web aplikaciju.

In [ ]:
import rag

rag.RAG_INSTRUKCIJE = RAG_INSTRUKCIJE      # moja pravila umjesto onih iz rag.py

index = rag.load_or_build_index(client, DATA_DIR, "index_moje.npz")
odgovor, izvori = rag.answer(client, index, "Moram li platiti papa test?")
print(odgovor)
print("Izvori:", izvori)

## 📝 Domaći zadatak: moj test set

Osam pitanja ima odgovor u bazi, dva nemaju.

| # | Pitanje | Očekivani odgovor | Izvor |
| --- | --- | --- | --- |
| 1 | Koliko dana traje normalan ciklus? | predvidiv, svakih 24 do 38 dana | figo-krvarenje.pdf |
| 2 | Koliko dugo smije trajati krvarenje? | do 8 dana, duže je produženo | figo-krvarenje.pdf |
| 3 | Koji su kriteriji za dijagnozu PCOS? | kriteriji iz smjernice iz 2023. | pcos-smjernica.pdf |
| 4 | Od koje godine se radi skrining grlića materice? | od 30. godine | who-skrining-grlic.pdf |
| 5 | Koliko minuta aktivnosti sedmično preporučuje WHO? | 150 do 300 umjerenog intenziteta | who-fizicka-aktivnost.pdf |
| 6 | Ko je oslobođen plaćanja participacije? | između ostalog trudnice, porodilje i žene na prevenciji karcinoma dojke i grlića materice | maksimalno-ucesce.pdf |
| 7 | Imam li pravo na izbor doktora? | da, izabrani doktor | zakon-zdravstvena-zastita.pdf |
| 8 | Pokriva li osnovni paket ginekologiju? | da, ginekologija i akušerstvo | osnovni-paket.pdf |
| 9 | Koliko košta pregled kod specijaliste? | **nema u bazi**, prilog s iznosima je skeniran | — |
| 10 | Koja je lista čekanja za mamografiju u Tuzli? | **nema u bazi** | — |

**Zašto baš ova dva negativna pitanja.** Nisu kao ono o pizzeriji. Pizzerija je očito izvan
teme, pa je sličnost niska i filter je lako odbije. Cijena pregleda i lista čekanja su **na
temu**, pa će pretraga naći chunkove koji djeluju srodno i sličnost neće biti niska. To je
pravi test praga: pitanje koje zvuči kao da pripada bazi, a odgovora u njoj nema.

Deveto pitanje je posljedica skeniranih stranica iz provjere na početku. Odgovor "nemam"
je tu **tačan odgovor**, a ne nedostatak.

In [ ]:
# Brza provjera test seta
test_set = [
    ("Koliko dana traje normalan menstrualni ciklus?", "figo-krvarenje.pdf"),
    ("Koliko dugo smije trajati menstrualno krvarenje?", "figo-krvarenje.pdf"),
    ("Koji su kriteriji za dijagnozu PCOS?", "pcos-smjernica.pdf"),
    ("Od koje godine se radi skrining grlića materice?", "who-skrining-grlic.pdf"),
    ("Koliko minuta aktivnosti sedmično preporučuje WHO?", "who-fizicka-aktivnost.pdf"),
    ("Ko je oslobođen plaćanja participacije?", "maksimalno-ucesce.pdf"),
    ("Imam li pravo na izbor doktora?", "zakon-zdravstvena-zastita.pdf"),
    ("Pokriva li osnovni paket ginekologiju?", "osnovni-paket.pdf"),
    ("Koliko košta pregled kod specijaliste?", None),
    ("Koja je lista čekanja za mamografiju u Tuzli?", None),
]

prosao = 0
for pitanje, ocekivani_izvor in test_set:
    odgovor = rag_answer(pitanje)
    najbolji = search(pitanje, k=1)[0]
    kaze_ne_zna = "Nemam tu informaciju" in odgovor

    if ocekivani_izvor is None:
        ok = kaze_ne_zna
    else:
        ok = (najbolji[1] == ocekivani_izvor) and not kaze_ne_zna
    prosao += ok

    print(f"[{'OK ' if ok else 'PAD'}] {pitanje}")
    print(f"      očekujem: {ocekivani_izvor or 'treba reći da ne zna'}")
    print(f"      našao: {najbolji[1]} ({najbolji[0]:.3f})")
    print(f"💬 {odgovor[:180]}...\n")

print(f"Prošlo: {prosao}/{len(test_set)}")

## 📝 Domaći zadatak

Napravi **test set od 10 pitanja** za bazu znanja Studentskog asistenta (ili za temu koja te zanima), u obliku:

| Pitanje | Očekivani odgovor | Izvor |
| --- | --- | --- |
| Koliko dug treba biti CV? | jedna stranica | pisanje_cv.md |

Uključi **bar 2 pitanja na koja odgovor NIJE u bazi**. Chatbot tada treba reći da ne zna. Test set koristimo na sljedećoj radionici za evaluaciju.

## Sažetak

| Pojam | Ukratko |
| --- | --- |
| RAG | Pretraga relevantnog teksta + generisanje odgovora na osnovu njega |
| Embedding | Vektor koji predstavlja značenje teksta |
| Kosinusna sličnost | Mjera koliko dva vektora "gledaju u istom smjeru" |
| Chunk | Manji dio dokumenta koji se zasebno pretražuje |
| Prag (MIN_SCORE) | Ispod njega smatramo da relevantan izvor ne postoji |

---

### Napomena o korištenju AI alata

Zadatke sam rješavala uz pomoć AI asistenta. Nije dobio gotova rješenja da ih prepiše,
nego smo išli zadatak po zadatak: ja sam govorila šta funkcija treba da radi i gdje mi
nije jasno, a onda sam svaki dio pokrenula i provjerila izlaz prije nego sam ga ostavila.

Gdje mi je AI najviše pomogao: `np.argsort(-scores)` za sortiranje po opadajućoj
sličnosti, i objašnjenje zašto je kod normalizovanih vektora skalarni proizvod već
kosinusna sličnost, pa se dijeljenje može preskočiti.

Gdje sam morala sama razmisliti: pomak kod chunkinga. Prvo sam mislila da se `start`
pomjera za `size`, ali tada nema preklapanja. Mora za `size - overlap`.